# Locally weighted regression (LWR)

_Machine Learning — more_

**Fit a fresh line at every query point, trusting nearby data most.**

Plain linear regression fits one line to all the data. That fails for curvy patterns.

     Locally weighted regression fits a fresh little line for each point you ask about.

---

This notebook is a practice scaffold for the **Locally weighted regression (LWR)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
rng = np.random.RandomState(0)

# Deterministic curvy data: y = sin(x) + 0.25x + noise.
x = np.linspace(-5, 5, 40)
y = np.sin(x) + 0.25*x + 0.15*rng.randn(40)
X = np.c_[np.ones_like(x), x]        # design matrix with intercept column

def lwr_predict(xq, tau):
    w = np.exp(-((x - xq)**2) / (2*tau**2))   # Gaussian weights
    W = np.diag(w)
    # Weighted normal equations: (X^T W X) theta = X^T W y
    theta = np.linalg.solve(X.T @ W @ X, X.T @ W @ y)
    return theta @ np.array([1.0, xq])

for tau in [0.3, 0.8, 3.0]:
    preds = [lwr_predict(xq, tau) for xq in [-3.0, 0.0, 3.0]]
    print("tau=%.1f  preds@(-3,0,3) = [%.3f, %.3f, %.3f]" %
          (tau, preds[0], preds[1], preds[2]))
print("small tau hugs the data (wiggly); large tau ~ one straight line")


## Visualize the data & results

_Can we fit a real noisy relationship without one global line?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

# REAL data: 442 diabetes patients, feature 2 = standardized BMI.
db = load_diabetes()
bmi = db.data[:, 2]
prog = db.target                         # disease progression after one year
X = np.c_[np.ones_like(bmi), bmi]        # design matrix with intercept

def lwr_predict(xq, tau=0.04):
    w = np.exp(-((bmi - xq)**2) / (2*tau**2))    # Gaussian weights near xq
    W = np.diag(w)
    A = X.T @ W @ X + 1e-6*np.eye(2)
    theta = np.linalg.solve(A, X.T @ W @ prog)   # weighted normal equations
    return float(theta @ np.array([1.0, xq]))

# show 60 patients to keep the scatter readable
rng = np.random.RandomState(0)
idx = np.sort(rng.choice(len(bmi), 60, replace=False))
grid = np.linspace(bmi.min(), bmi.max(), 50)
fit = [lwr_predict(q) for q in grid]

plt.scatter(bmi[idx], prog[idx], color='#4ea1ff', label='442 real patients (60 shown)')
plt.plot(grid, fit, color='#ffb454', label='LWR fit (refit a weighted line at every BMI)')
plt.xlabel('body mass index (standardized)')
plt.ylabel('disease progression after one year')
plt.title('Diabetes: BMI vs disease progression with a locally-weighted fit')
plt.legend(); plt.show()
